# 03 Matrix Profile Motif Visual Study

## Objective
Create the main Matrix Profile study notebook for visualizing motif distances, empirical thresholds, top motifs, motif overlays, motif timelines, evaluation metrics, and runtime evidence from already-computed files.

## Input files
- `results/motifs/matrix_profile/matrix_profile_motif_results.parquet`
- `results/motifs/matrix_profile/matrix_profile_evaluation.parquet`
- `results/motifs/matrix_profile/matrix_profile_runtime.parquet`
- `results/motifs/matrix_profile/matrix_profile_profiles.parquet`
- Thesis-scope feature parquet files under `final_dataset/features`

## Output folder
`reports/study_notebooks/figures/matrix_profile` and `reports/study_notebooks/tables`.

## Thesis relevance
This notebook explains what Matrix Profile motif distances mean, derives empirical thresholds from existing distances, and visualizes agnostic vs regime-conditioned motif discovery without rerunning Matrix Profile.

## Analysis-only safety
This notebook never imports or calls STUMPY, STUMP/MSTUMP, HMM fitting, LoCoMotif search, or any other expensive experiment algorithm. It only reads saved result files and thesis-scope feature parquet files, then produces derived tables and figures.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "HPC workflow" / "HPC_Regime_and_motif_discovery" / "notebooks" / "study"
if NOTEBOOK_DIR.exists() and str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from study_helpers import *

ensure_study_output_dirs()
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Project root:", PROJECT_ROOT)
print("Workflow root:", WORKFLOW_ROOT)
print("Study outputs:", REPORT_ROOT)


## Load Matrix Profile Results


In [ ]:
mp_dir = result_path("motifs", "matrix_profile")
mp_paths = {
    "motif_results": resolve_existing_file(mp_dir, "matrix_profile_motif_results.parquet"),
    "evaluation": resolve_existing_file(mp_dir, "matrix_profile_evaluation.parquet"),
    "runtime": resolve_existing_file(mp_dir, "matrix_profile_runtime.parquet"),
    "profiles": resolve_existing_file(mp_dir, "matrix_profile_profiles.parquet"),
}
mp = safe_read_parquet(mp_paths["motif_results"])
eval_df = safe_read_parquet(mp_paths["evaluation"])
runtime = safe_read_parquet(mp_paths["runtime"])
profiles = safe_read_parquet(mp_paths["profiles"])
for col in ["motif_timestamp_1", "motif_timestamp_2", "motif_end_timestamp_1", "motif_end_timestamp_2"]:
    if col in mp.columns:
        mp[col] = pd.to_datetime(mp[col], errors="coerce")
distance_col = find_distance_column(mp)
print("Selected distance column:", distance_col)


## File Inventory and Schema


In [ ]:
inventory_rows = []
for name, path in mp_paths.items():
    df = {"motif_results": mp, "evaluation": eval_df, "runtime": runtime, "profiles": profiles}[name]
    inventory_rows.append({
        "file": name,
        "path": str(path),
        "exists": path.exists(),
        "rows": len(df),
        "columns": len(df.columns),
        "useful_columns": useful_columns(df),
    })
inventory = pd.DataFrame(inventory_rows)
display_table(inventory)
save_table(inventory, "study_mp_file_inventory")


## Experiment Scope and Motif Result Overview


In [ ]:
overview_tables = {
    "study_mp_rows_by_asset_frequency_mode": group_count_table(mp, ["asset", "frequency", "mode"], "study_mp_rows_by_asset_frequency_mode"),
    "study_mp_rows_by_regime_method": group_count_table(mp, ["regime_method"], "study_mp_rows_by_regime_method"),
    "study_mp_rows_by_regime_label": group_count_table(mp, ["regime_label"], "study_mp_rows_by_regime_label"),
    "study_mp_rows_by_frequency_window": group_count_table(mp, ["frequency", "window_length"], "study_mp_rows_by_frequency_window"),
    "study_mp_rows_by_feature_set": group_count_table(mp, ["feature_set"], "study_mp_rows_by_feature_set"),
    "study_mp_rows_by_profile_type": group_count_table(mp, ["profile_type"], "study_mp_rows_by_profile_type"),
    "study_mp_rows_by_asset_frequency_regime": group_count_table(mp, ["asset", "frequency", "regime_label"], "study_mp_rows_by_asset_frequency_regime"),
}
for name, table in overview_tables.items():
    print("\n", name)
    display_table(table, 15)
summary = pd.DataFrame([{"metric": "total_motif_rows", "value": len(mp)}])
save_table(summary, "study_mp_total_motif_rows")


## Motif Distances and Empirical Thresholds


In [ ]:
def plot_distance_hist(df, title, filename, by=None):
    if df.empty or not distance_col:
        print(f"No distance data for {title}")
        return
    fig, ax = plt.subplots(figsize=(12, 6))
    if by and by in df.columns:
        for key, part in df.groupby(by, dropna=False):
            values = pd.to_numeric(part[distance_col], errors="coerce").dropna()
            if len(values):
                ax.hist(values, bins=35, alpha=0.45, label=str(key), density=False)
        ax.legend(title=by)
    else:
        pd.to_numeric(df[distance_col], errors="coerce").dropna().plot(kind="hist", bins=50, ax=ax, color="#4C78A8")
    ax.set_title(title)
    ax.set_xlabel(distance_col)
    ax.set_ylabel("Motif rows")
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["matrix_profile"])
    plt.show()

plot_distance_hist(mp, "Matrix Profile motif distance distribution, overall", "study_mp_distance_distribution_overall")
plot_distance_hist(mp, "Matrix Profile distances: agnostic vs conditioned", "study_mp_distance_distribution_by_mode", by="mode")
plot_distance_hist(mp, "Matrix Profile distances by regime label", "study_mp_distance_distribution_by_regime_label", by="regime_label")
plot_distance_hist(mp, "Matrix Profile distances by window length", "study_mp_distance_distribution_by_window_length", by="window_length")
plot_distance_hist(mp, "Matrix Profile distances by feature set", "study_mp_distance_distribution_by_feature_set", by="feature_set")


In [ ]:
def threshold_record(scope, df):
    if df.empty or not distance_col:
        return {"scope": scope, "rows": len(df)}
    values = pd.to_numeric(df[distance_col], errors="coerce").dropna()
    if values.empty:
        return {"scope": scope, "rows": len(df)}
    qs = values.quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75])
    return {
        "scope": scope,
        "rows": len(values),
        "q01": qs.loc[0.01],
        "q05": qs.loc[0.05],
        "q10": qs.loc[0.10],
        "q25": qs.loc[0.25],
        "median": qs.loc[0.50],
        "q75": qs.loc[0.75],
        "min": values.min(),
        "max": values.max(),
    }

scopes = [("overall", mp)]
if "mode" in mp.columns:
    scopes.extend([(str(k), v) for k, v in mp.groupby("mode", dropna=False)])
if "regime_label" in mp.columns:
    for label in ["low_vol", "high_vol", "extreme_vol"]:
        scopes.append((label, mp[mp["regime_label"].astype(str).eq(label)]))

thresholds = pd.DataFrame([threshold_record(name, df) for name, df in scopes])
display_table(thresholds, 20)
save_table(thresholds, "study_mp_empirical_distance_thresholds")

if not mp.empty and distance_col:
    values = pd.to_numeric(mp[distance_col], errors="coerce").dropna()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(values, bins=60, color="#4C78A8", alpha=0.75)
    overall = thresholds[thresholds["scope"].eq("overall")]
    for col, color in [("q01", "#D62728"), ("q05", "#FF7F0E"), ("q10", "#2CA02C"), ("median", "#000000")]:
        if col in overall.columns and not overall[col].isna().all():
            val = overall[col].iloc[0]
            ax.axvline(val, color=color, linestyle="--", linewidth=1.5, label=col)
    ax.set_title("Matrix Profile empirical distance thresholds")
    ax.set_xlabel(distance_col)
    ax.set_ylabel("Motif rows")
    ax.legend()
    fig.tight_layout()
    save_fig(fig, "study_mp_distance_threshold_lines", FIGURE_DIRS["matrix_profile"])
    plt.show()


### Interpretation
Matrix Profile distances are lower-is-better for the selected distance column. Matrix Profile does not provide one universal intrinsic motif threshold, so this notebook reports empirical thresholds from the observed result distribution: top 1%, top 5%, top 10%, quartiles, median, minimum, and maximum.


## Top 10 and Top 20 Motifs


In [ ]:
top_cols_preferred = [
    "asset", "frequency", "mode", "regime_method", "regime_label", "feature_set", "profile_type",
    "window_length", "motif_rank", "motif_start_1", "motif_start_2", "motif_timestamp_1",
    "motif_timestamp_2", "segment_id", "used_gpu",
]
if distance_col:
    top_cols_preferred.insert(-2, distance_col)
top_cols = [c for c in top_cols_preferred if c in mp.columns]

def top_table(df, n, name):
    if df.empty or not distance_col:
        print(f"No top table for {name}")
        return pd.DataFrame()
    out = df.sort_values(distance_col, ascending=True).head(n)[top_cols].copy()
    save_table(out, name)
    display_table(out, n)
    return out

top10_overall = top_table(mp, 10, "study_mp_top10_overall")
top20_overall = top_table(mp, 20, "study_mp_top20_overall")
top10_agnostic = top_table(mp[mp["mode"].astype(str).eq("agnostic")] if "mode" in mp.columns else pd.DataFrame(), 10, "study_mp_top10_agnostic")
top10_conditioned = top_table(mp[mp["mode"].astype(str).eq("conditioned")] if "mode" in mp.columns else pd.DataFrame(), 10, "study_mp_top10_conditioned")
for label in ["low_vol", "high_vol", "extreme_vol"]:
    subset = mp[mp["regime_label"].astype(str).eq(label)] if "regime_label" in mp.columns else pd.DataFrame()
    top_table(subset, 10, f"study_mp_top10_{label}")
for asset in ["BTCUSDT", "ETHUSDT"]:
    subset = mp[(mp["asset"].astype(str).eq(asset)) & (mp["frequency"].astype(str).eq("15m"))] if {"asset", "frequency"}.issubset(mp.columns) else pd.DataFrame()
    top_table(subset, 10, f"study_mp_top10_{asset}_15m")


## Top Motif Overlay Plots


In [ ]:
def plot_motif_overlay(row, filename):
    asset = str(row.get("asset", ""))
    frequency = str(row.get("frequency", ""))
    feature_set = str(row.get("feature_set", "close"))
    window = int(row.get("window_length", 0)) if pd.notna(row.get("window_length", np.nan)) else 0
    s1 = int(row.get("motif_start_1", -1)) if pd.notna(row.get("motif_start_1", np.nan)) else -1
    s2 = int(row.get("motif_start_2", -1)) if pd.notna(row.get("motif_start_2", np.nan)) else -1
    if window <= 1 or s1 < 0 or s2 < 0:
        print("Cannot plot overlay: missing start indices or window length.")
        return False
    feature = load_feature_data(asset, frequency)
    value_col = find_feature_value_column(feature, feature_set)
    if feature.empty or not value_col:
        print(f"Cannot plot overlay for {asset} {frequency}: feature column unavailable.")
        return False
    if s1 + window > len(feature) or s2 + window > len(feature):
        print(f"Cannot plot overlay for {asset} {frequency}: motif windows exceed feature length.")
        return False
    y1 = z_normalize(feature[value_col].iloc[s1:s1 + window])
    y2 = z_normalize(feature[value_col].iloc[s2:s2 + window])
    x = np.arange(window)
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(x, y1, label=f"window 1 @ {s1}", linewidth=2.0, color="#1F77B4")
    ax.plot(x, y2, label=f"nearest window @ {s2}", linewidth=2.0, color="#FF7F0E")
    dist_text = f"{distance_col}={row.get(distance_col):.4g}" if distance_col and pd.notna(row.get(distance_col, np.nan)) else "distance unavailable"
    title_bits = [asset, frequency, str(row.get("mode", "")), str(row.get("regime_label", "")), value_col, f"w={window}", dist_text]
    ax.set_title(" | ".join([b for b in title_bits if b and b != "nan"]))
    ax.set_xlabel("Window offset")
    ax.set_ylabel("z-normalized value")
    ax.legend()
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["matrix_profile"])
    plt.show()
    return True

def overlay_batch(df, label, n=5):
    if df.empty or not distance_col:
        print(f"No rows available for overlay batch: {label}")
        return
    selected = df.sort_values(distance_col, ascending=True).head(n)
    successes = 0
    for idx, row in selected.iterrows():
        name = f"study_mp_overlay_{label}_{successes + 1}_{row.get('asset','asset')}_{row.get('frequency','freq')}_{row.get('feature_set','feature')}_w{row.get('window_length','w')}"
        successes += int(plot_motif_overlay(row, name))
    print(f"{label}: created {successes} overlay plots.")

overlay_batch(mp, "top5_overall")
if "mode" in mp.columns:
    overlay_batch(mp[mp["mode"].astype(str).eq("agnostic")], "top5_agnostic")
    overlay_batch(mp[mp["mode"].astype(str).eq("conditioned")], "top5_conditioned")
if "regime_label" in mp.columns:
    overlay_batch(mp[mp["regime_label"].astype(str).eq("high_vol")], "top5_high_vol")
    overlay_batch(mp[mp["regime_label"].astype(str).eq("low_vol")], "top5_low_vol")


### Interpretation
Each overlay compares the motif window and its nearest-neighbour window after z-normalization. Similar shapes with low distances are candidate recurring subsequences under the selected feature, window length, and regime context.


## Matrix Profile Curves or Ranked Distance Plots


In [ ]:
def plot_profile_for_top(row, filename):
    if profiles.empty:
        return False
    filters = {}
    for col in ["asset", "frequency", "mode", "regime_method", "regime_label", "segment_id", "window_length", "feature_set"]:
        if col in profiles.columns and col in row.index and pd.notna(row[col]):
            filters[col] = row[col]
    work = profiles.copy()
    for col, val in filters.items():
        work = work[work[col].astype(str) == str(val)]
    value_col = "matrix_profile" if "matrix_profile" in work.columns else find_distance_column(work)
    idx_col = "profile_index" if "profile_index" in work.columns else None
    if work.empty or not value_col:
        return False
    if len(work) > 20000:
        work = work.sample(20000, random_state=7).sort_values(idx_col) if idx_col else work.sample(20000, random_state=7)
    fig, ax = plt.subplots(figsize=(13, 5))
    x = work[idx_col] if idx_col else np.arange(len(work))
    ax.plot(x, work[value_col], color="#4C78A8", linewidth=1.0)
    for start_col, color in [("motif_start_1", "#D62728"), ("motif_start_2", "#FF7F0E")]:
        if start_col in row.index and pd.notna(row[start_col]):
            ax.axvline(float(row[start_col]), color=color, linestyle="--", label=start_col)
    if distance_col and not thresholds.empty:
        overall = thresholds[thresholds["scope"].eq("overall")]
        for q in ["q01", "q05", "q10"]:
            if q in overall.columns and not overall[q].isna().all():
                ax.axhline(overall[q].iloc[0], linestyle=":", linewidth=1.2, label=q)
    ax.set_title("Matrix Profile curve with top motif locations")
    ax.set_xlabel(idx_col or "Profile row")
    ax.set_ylabel(value_col)
    ax.legend()
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["matrix_profile"])
    plt.show()
    return True

profile_done = False
if not top10_overall.empty:
    profile_done = plot_profile_for_top(top10_overall.iloc[0], "study_mp_profile_curve_top_motif")

if not profile_done:
    print("Raw matrix profile curve unavailable or not alignable; plotting ranked distances instead.")
    if not mp.empty and distance_col:
        ranked = mp.sort_values(distance_col, ascending=True).reset_index(drop=True)
        ranked["rank"] = np.arange(1, len(ranked) + 1)
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(ranked["rank"], ranked[distance_col], marker=".", linewidth=1.0)
        ax.set_title("Ranked Matrix Profile motif distances")
        ax.set_xlabel("Rank, lowest distance first")
        ax.set_ylabel(distance_col)
        fig.tight_layout()
        save_fig(fig, "study_mp_ranked_distance_elbow", FIGURE_DIRS["matrix_profile"])
        plt.show()


## Overlapping Time Motif Illustrations


In [ ]:
def overlap_table(df, n=50):
    if df.empty:
        return pd.DataFrame()
    rows = []
    for i, row in df.head(n).iterrows():
        w = int(row.get("window_length", 0)) if pd.notna(row.get("window_length", np.nan)) else 0
        s1 = int(row.get("motif_start_1", -1)) if pd.notna(row.get("motif_start_1", np.nan)) else -1
        s2 = int(row.get("motif_start_2", -1)) if pd.notna(row.get("motif_start_2", np.nan)) else -1
        if w <= 0 or s1 < 0 or s2 < 0:
            continue
        a1, a2 = s1, s1 + w
        b1, b2 = s2, s2 + w
        overlap_len = max(0, min(a2, b2) - max(a1, b1))
        rows.append({
            "motif_id": i,
            "asset": row.get("asset"),
            "frequency": row.get("frequency"),
            "mode": row.get("mode"),
            "regime_label": row.get("regime_label"),
            "window_length": w,
            "start_1": s1,
            "start_2": s2,
            "overlap_length": overlap_len,
            "overlap_ratio": overlap_len / w if w else np.nan,
        })
    return pd.DataFrame(rows)

mp_top50 = mp.sort_values(distance_col, ascending=True).head(50) if distance_col else mp.head(50)
overlaps = overlap_table(mp_top50)
display_table(overlaps, 20)
save_table(overlaps, "study_mp_top50_motif_window_overlap")

def plot_motif_timeline(df, asset, frequency, filename, title_extra=""):
    subset = filter_scope(df, asset, frequency)
    if subset.empty:
        print(f"No motif rows for timeline: {asset} {frequency}")
        return
    if distance_col:
        subset = subset.sort_values(distance_col, ascending=True).head(20)
    feature = load_feature_data(asset, frequency)
    ts = timestamp_column(feature)
    fig, ax = plt.subplots(figsize=(14, 5))
    if not feature.empty and ts and "close" in feature.columns:
        sampled = feature[[ts, "close"]].dropna().sort_values(ts)
        if len(sampled) > 8000:
            sampled = sampled.iloc[:: int(np.ceil(len(sampled) / 8000))]
        ax.plot(sampled[ts], sampled["close"], color="0.75", linewidth=0.8, label="close")
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    for j, (_, row) in enumerate(subset.iterrows()):
        w = int(row.get("window_length", 1)) if pd.notna(row.get("window_length", np.nan)) else 1
        for start_col, time_col in [("motif_start_1", "motif_timestamp_1"), ("motif_start_2", "motif_timestamp_2")]:
            start_time = row.get(time_col)
            if pd.isna(start_time) and not feature.empty and ts and pd.notna(row.get(start_col, np.nan)):
                idx = int(row[start_col])
                if 0 <= idx < len(feature):
                    start_time = feature.iloc[idx][ts]
            if pd.notna(start_time):
                start_time = pd.to_datetime(start_time)
                end_time = start_time + pd.Timedelta(minutes=15 * w if frequency == "15m" else 60 * w)
                ax.axvspan(start_time, end_time, color=colors[j % len(colors)], alpha=0.18)
    ax.set_title(f"{asset} {frequency} top motif timeline {title_extra}".strip())
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Close price or motif spans")
    fig.autofmt_xdate()
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["matrix_profile"])
    plt.show()

plot_motif_timeline(mp, "BTCUSDT", "15m", "study_mp_timeline_BTCUSDT_15m_top_motifs")
plot_motif_timeline(mp, "ETHUSDT", "15m", "study_mp_timeline_ETHUSDT_15m_top_motifs")
plot_motif_timeline(mp, "BTCUSDT", "1h", "study_mp_timeline_BTCUSDT_1h_top_motifs")
if "regime_label" in mp.columns:
    plot_motif_timeline(mp[mp["regime_label"].astype(str).eq("high_vol")], "BTCUSDT", "15m", "study_mp_timeline_BTCUSDT_15m_high_vol_top_motifs", "high_vol")


## Agnostic vs Conditioned Evidence


In [ ]:
def mode_metrics(df, mode):
    part = df[df["mode"].astype(str).eq(mode)] if "mode" in df.columns else pd.DataFrame()
    values = pd.to_numeric(part[distance_col], errors="coerce").dropna() if distance_col and not part.empty else pd.Series(dtype=float)
    return {
        "saved motif rows": len(part),
        "median distance": values.median() if len(values) else np.nan,
        "q10 distance": values.quantile(0.10) if len(values) else np.nan,
        "best distance": values.min() if len(values) else np.nan,
        "number of regimes represented": part["regime_label"].nunique(dropna=True) if "regime_label" in part.columns else np.nan,
        "number of windows represented": part["window_length"].nunique(dropna=True) if "window_length" in part.columns else np.nan,
        "number of feature sets represented": part["feature_set"].nunique(dropna=True) if "feature_set" in part.columns else np.nan,
        "runtime if available": part["runtime_seconds"].sum() if "runtime_seconds" in part.columns else np.nan,
        "figure examples available": "generated above when feature windows align",
    }

agnostic = mode_metrics(mp, "agnostic")
conditioned = mode_metrics(mp, "conditioned")
interpretations = {
    "saved motif rows": "Conditioned row counts are expected to be larger when searches are repeated across regimes, methods, and segments.",
    "median distance": "Lower values indicate closer motif-neighbour pairs within the saved results.",
    "q10 distance": "Lower top-decile distances indicate stronger best-candidate similarity.",
    "best distance": "Best observed saved motif distance.",
    "number of regimes represented": "Conditioned discovery attaches motifs to market states.",
    "number of windows represented": "Window-length coverage in the saved benchmark.",
    "number of feature sets represented": "Feature coverage in the saved benchmark.",
    "runtime if available": "Runtime is read from saved result rows when present.",
    "figure examples available": "Visual support depends on available feature data and start indices.",
}
comparison_rows = []
for metric in agnostic:
    comparison_rows.append({
        "Metric": metric,
        "Agnostic": agnostic[metric],
        "Conditioned": conditioned[metric],
        "Interpretation": interpretations.get(metric, ""),
    })
mode_comparison = pd.DataFrame(comparison_rows)
display_table(mode_comparison, 20)
save_table(mode_comparison, "study_mp_agnostic_vs_conditioned_evidence")


### Interpretation
Conditioned discovery increases regime-specific interpretability by attaching motif candidates to market states. Row counts are larger because the search is repeated across regimes, regime methods, and segments.


## Evaluation Metrics


In [ ]:
display_table(eval_df, 20)
save_table(eval_df, "study_mp_evaluation_raw")

numeric_metrics = [c for c in eval_df.columns if pd.api.types.is_numeric_dtype(eval_df[c])]
group_sets = [
    ["mode"],
    ["asset", "frequency", "mode"],
    ["regime_label"],
    ["window_length"],
    ["feature_set"],
    ["profile_type"],
]
for group_cols in group_sets:
    existing = [c for c in group_cols if c in eval_df.columns]
    if eval_df.empty or not existing or not numeric_metrics:
        continue
    summary = eval_df.groupby(existing, dropna=False)[numeric_metrics].agg(["mean", "median", "min", "max"])
    summary.columns = ["_".join(col).strip("_") for col in summary.columns.to_flat_index()]
    summary = summary.reset_index()
    name = "study_mp_evaluation_by_" + "_".join(existing)
    display_table(summary, 20)
    save_table(summary, name)


## Runtime and CPU/GPU Audit


In [ ]:
display_table(runtime, 20)
save_table(runtime, "study_mp_runtime_raw")
for group_cols in [["asset", "frequency", "mode", "profile_type"], ["window_length"], ["profile_type"]]:
    existing = [c for c in group_cols if c in runtime.columns]
    if runtime.empty or "runtime_seconds" not in runtime.columns or not existing:
        continue
    out = runtime.groupby(existing, dropna=False)["runtime_seconds"].agg(["count", "sum", "mean", "median", "min", "max"]).reset_index()
    display_table(out, 20)
    save_table(out, "study_mp_runtime_by_" + "_".join(existing))

if "used_gpu" in mp.columns:
    gpu_audit = mp["used_gpu"].value_counts(dropna=False).rename_axis("used_gpu").reset_index(name="rows")
    display_table(gpu_audit)
    save_table(gpu_audit, "study_mp_gpu_audit")
    if len(gpu_audit) and gpu_audit["used_gpu"].astype(str).str.lower().isin(["false", "0"]).all():
        print("All reported Matrix Profile results were generated using CPU execution. GPU acceleration was allowed by the pipeline but was not used in the executed run.")
else:
    print("No used_gpu column is available in Matrix Profile motif results.")


## Key findings
Use the generated top motif tables, threshold table, overlays, timelines, evaluation summaries, and runtime summaries to answer which Matrix Profile motifs were found and where the best distances occur.

## Thesis-safe interpretation
Matrix Profile found fixed-length motif candidates in the saved results when motif rows and distance columns are present. Lower distance values indicate stronger shape similarity under the selected feature and window length. Empirical thresholds are descriptive quantiles of the saved result distribution, not universal Matrix Profile constants.

## Limitations
This notebook does not rerun Matrix Profile and cannot recover missing motif windows, missing feature files, or empty profile tables. Agnostic and conditioned searches differ in scope, so row counts alone are not evidence of superiority.

## Recommended figures for thesis
- Distance distribution with empirical thresholds
- Top motif overlays for BTCUSDT 15m
- Top motif timelines for BTCUSDT 15m and ETHUSDT 15m
- Agnostic vs conditioned evidence table
- Runtime and CPU/GPU audit table
